In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the four supermarket datasets

transactions = pd.read_csv("/content/transactions.csv")
products = pd.read_csv("/content/products.csv")
branches = pd.read_csv("/content/branches.csv")
customers = pd.read_csv("/content/customers.csv")

# Display the number of rows and columns in each dataset

print("Transactions:", transactions.shape)
print("Products:", products.shape)
print("Branches:", branches.shape)
print("Customers:", customers.shape)

Transactions: (50000, 8)
Products: (20, 4)
Branches: (5, 2)
Customers: (5000, 2)


In [3]:
# 1. Check data types
print("DATA TYPES")
transactions.info()

# 2. Check missing values
print("\nMISSING VALUES")
print(transactions.isnull().sum())

# 3. Check duplicate transaction IDs
print("\nDUPLICATE TRANSACTION IDs")
print(transactions["transaction_id"].duplicated().sum())

# 4. Check invalid quantities and prices
print("\nINVALID VALUES")
print("Quantities equal to or below zero:",
      (transactions["quantity"] <= 0).sum())

print("Prices equal to or below zero:",
      (transactions["unit_price"] <= 0).sum())

# 5. Check the date range
transactions["date"] = pd.to_datetime(
    transactions["date"], errors="coerce"
)

print("\nDATE RANGE")
print("Earliest date:", transactions["date"].min())
print("Latest date:", transactions["date"].max())

# 6. Check the dataset structure
print("\nFIRST FIVE TRANSACTIONS")
display(transactions.head())

DATA TYPES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaction_id  50000 non-null  int64 
 1   date            50000 non-null  object
 2   customer_id     50000 non-null  int64 
 3   product_id      50000 non-null  int64 
 4   branch_id       50000 non-null  int64 
 5   quantity        50000 non-null  int64 
 6   unit_price      50000 non-null  int64 
 7   sales_amount    50000 non-null  int64 
dtypes: int64(7), object(1)
memory usage: 3.1+ MB

MISSING VALUES
transaction_id    0
date              0
customer_id       0
product_id        0
branch_id         0
quantity          0
unit_price        0
sales_amount      0
dtype: int64

DUPLICATE TRANSACTION IDs
0

INVALID VALUES
Quantities equal to or below zero: 0
Prices equal to or below zero: 0

DATE RANGE
Earliest date: 2025-01-01 00:00:00
Latest date: 2025-12-31 00:00:00

FIRST 

,transaction_id,date,customer_id,product_id,branch_id,quantity,unit_price,sales_amount
0,1,2025-02-20,3989,19,4,9,40,360
1,2,2025-03-12,3550,1,1,2,180,360
2,3,2025-02-17,1122,10,1,2,300,600
3,4,2025-01-19,282,11,4,4,100,400
4,5,2025-09-19,630,12,4,1,150,150


In [4]:
# Clean the supermarket transaction data

def clean_transactions(raw_data):
    clean_data = raw_data.copy()
    cleaning_log = []

    # Record initial row count
    original_rows = len(clean_data)

    cleaning_log.append({
        "Action": "Initial row count",
        "Rows Before": original_rows,
        "Rows After": original_rows,
        "Rows Removed": 0
    })

    # 1. Remove duplicate transaction IDs
    before = len(clean_data)
    clean_data = clean_data.drop_duplicates(
        subset=["transaction_id"]
    )

    cleaning_log.append({
        "Action": "Remove duplicate transaction IDs",
        "Rows Before": before,
        "Rows After": len(clean_data),
        "Rows Removed": before - len(clean_data)
    })

    # 2. Convert dates to datetime
    clean_data["date"] = pd.to_datetime(
        clean_data["date"], errors="coerce"
    )

    # 3. Remove rows with missing essential values
    before = len(clean_data)

    essential_columns = [
        "transaction_id", "date", "customer_id",
        "product_id", "branch_id", "quantity", "unit_price"
    ]

    clean_data = clean_data.dropna(subset=essential_columns)

    cleaning_log.append({
        "Action": "Remove missing essential values",
        "Rows Before": before,
        "Rows After": len(clean_data),
        "Rows Removed": before - len(clean_data)
    })

    # 4. Remove invalid quantities and prices
    before = len(clean_data)

    clean_data = clean_data[
        (clean_data["quantity"] > 0) &
        (clean_data["unit_price"] > 0)
    ].copy()

    cleaning_log.append({
        "Action": "Remove invalid quantities or prices",
        "Rows Before": before,
        "Rows After": len(clean_data),
        "Rows Removed": before - len(clean_data)
    })

    # 5. Recalculate sales amounts
    clean_data["sales_amount"] = (
        clean_data["quantity"] * clean_data["unit_price"]
    )

    cleaning_log.append({
        "Action": "Recalculate sales amounts",
        "Rows Before": len(clean_data),
        "Rows After": len(clean_data),
        "Rows Removed": 0
    })

    # 6. Reset row numbering
    clean_data = clean_data.reset_index(drop=True)

    return clean_data, cleaning_log


# Apply the cleaning function
clean_transactions_data, cleaning_log = clean_transactions(transactions)

# Display cleaning results
print("Original transaction rows:", len(transactions))
print("Clean transaction rows:", len(clean_transactions_data))

print("\nCLEANING LOG")
display(pd.DataFrame(cleaning_log))

print("\nCLEANED DATA PREVIEW")
display(clean_transactions_data.head())

Original transaction rows: 50000
Clean transaction rows: 50000

CLEANING LOG


,Action,Rows Before,Rows After,Rows Removed
0,Initial row count,50000,50000,0
1,Remove duplicate transaction IDs,50000,50000,0
2,Remove missing essential values,50000,50000,0
3,Remove invalid quantities or prices,50000,50000,0
4,Recalculate sales amounts,50000,50000,0



CLEANED DATA PREVIEW


,transaction_id,date,customer_id,product_id,branch_id,quantity,unit_price,sales_amount
0,1,2025-02-20,3989,19,4,9,40,360
1,2,2025-03-12,3550,1,1,2,180,360
2,3,2025-02-17,1122,10,1,2,300,600
3,4,2025-01-19,282,11,4,4,100,400
4,5,2025-09-19,630,12,4,1,150,150


In [5]:
# Create an SQLite database
connection = sqlite3.connect("/content/supermarket.db")

# Save the cleaned transactions and supporting tables
clean_transactions_data.to_sql(
    "transactions", connection, if_exists="replace", index=False
)

products.to_sql(
    "products", connection, if_exists="replace", index=False
)

branches.to_sql(
    "branches", connection, if_exists="replace", index=False
)

customers.to_sql(
    "customers", connection, if_exists="replace", index=False
)

print("All four datasets have been saved to SQLite.")

# Check the number of records in each table
for table in ["transactions", "products", "branches", "customers"]:
    result = pd.read_sql(
        f"SELECT COUNT(*) AS total_records FROM {table}",
        connection
    )
    print(f"\n{table.capitalize()}:")
    display(result)

All four datasets have been saved to SQLite.

Transactions:


,total_records
0,50000



Products:


,total_records
0,20



Branches:


,total_records
0,5



Customers:


,total_records
0,5000


In [6]:
# Six-dimension data-quality audit

audit_results = [
    {
        "Dimension": "Completeness",
        "Check": "Missing values in essential transaction fields",
        "Result": int(
            clean_transactions_data[
                ["transaction_id", "date", "customer_id",
                 "product_id", "branch_id", "quantity", "unit_price"]
            ].isnull().sum().sum()
        )
    },
    {
        "Dimension": "Uniqueness",
        "Check": "Duplicate transaction IDs",
        "Result": int(
            clean_transactions_data["transaction_id"].duplicated().sum()
        )
    },
    {
        "Dimension": "Validity",
        "Check": "Transactions with non-positive quantities or prices",
        "Result": int(
            (
                (clean_transactions_data["quantity"] <= 0) |
                (clean_transactions_data["unit_price"] <= 0)
            ).sum()
        )
    },
    {
        "Dimension": "Consistency",
        "Check": "Sales amounts that do not match quantity × unit price",
        "Result": int(
            (
                clean_transactions_data["sales_amount"]
                != clean_transactions_data["quantity"]
                * clean_transactions_data["unit_price"]
            ).sum()
        )
    },
    {
        "Dimension": "Accuracy",
        "Check": "Transactions referencing unknown products",
        "Result": int(
            (~clean_transactions_data["product_id"].isin(
                products["product_id"]
            )).sum()
        )
    },
    {
        "Dimension": "Timeliness",
        "Check": "Transactions with dates outside the expected year 2025",
        "Result": int(
            (
                (clean_transactions_data["date"].dt.year < 2025) |
                (clean_transactions_data["date"].dt.year > 2025)
            ).sum()
        )
    }
]

audit_df = pd.DataFrame(audit_results)

display(audit_df)

,Dimension,Check,Result
0,Completeness,Missing values in essential transaction fields,0
1,Uniqueness,Duplicate transaction IDs,0
2,Validity,Transactions with non-positive quantities or p...,0
3,Consistency,Sales amounts that do not match quantity × uni...,0
4,Accuracy,Transactions referencing unknown products,0
5,Timeliness,Transactions with dates outside the expected y...,0
